Собрать инфу про каждого гонщика по ходу всего сезона

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os


os.makedirs("cache", exist_ok=True)

fastf1.Cache.enable_cache("cache")



In [2]:
schedule = fastf1.get_event_schedule(2024)

schedule[
    ["RoundNumber", "EventName"]
].head(11)

,RoundNumber,EventName
0,0,Pre-Season Testing
1,1,Bahrain Grand Prix
2,2,Saudi Arabian Grand Prix
3,3,Australian Grand Prix
4,4,Japanese Grand Prix
5,5,Chinese Grand Prix
6,6,Miami Grand Prix
7,7,Emilia Romagna Grand Prix
8,8,Monaco Grand Prix
9,9,Canadian Grand Prix


In [3]:
def load_race_results(year, round_number):
    session = fastf1.get_session(year, round_number, "R")
    session.load()

    results = session.results.copy()

    results["Year"] = year
    results["Round"] = round_number

    return results

race = load_race_results(2024, 1)

race.head()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Year,Round
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671c6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,NaT,NaT,NaT,0 days 01:31:44.742000,Finished,26.0,57.0,2024,1
11,11,S PEREZ,PER,perez,Red Bull Racing,3671c6,red_bull,Sergio,Perez,Sergio Perez,...,5.0,NaT,NaT,NaT,0 days 00:00:22.457000,Finished,18.0,57.0,2024,1
55,55,C SAINZ,SAI,sainz,Ferrari,e8002d,ferrari,Carlos,Sainz,Carlos Sainz,...,4.0,NaT,NaT,NaT,0 days 00:00:25.110000,Finished,15.0,57.0,2024,1
16,16,C LECLERC,LEC,leclerc,Ferrari,e8002d,ferrari,Charles,Leclerc,Charles Leclerc,...,2.0,NaT,NaT,NaT,0 days 00:00:39.669000,Finished,12.0,57.0,2024,1
63,63,G RUSSELL,RUS,russell,Mercedes,27f4d2,mercedes,George,Russell,George Russell,...,3.0,NaT,NaT,NaT,0 days 00:00:46.788000,Finished,10.0,57.0,2024,1


In [4]:
race.columns.tolist()

columns = [
    "Year",
    "Round",
    "Abbreviation",
    "FullName",
    "TeamName",
    "GridPosition",
    "Position",
    "Status",
    "Points"
]

race[columns].head(20)




,Year,Round,Abbreviation,FullName,TeamName,GridPosition,Position,Status,Points
1,2024,1,VER,Max Verstappen,Red Bull Racing,1.0,1.0,Finished,26.0
11,2024,1,PER,Sergio Perez,Red Bull Racing,5.0,2.0,Finished,18.0
55,2024,1,SAI,Carlos Sainz,Ferrari,4.0,3.0,Finished,15.0
16,2024,1,LEC,Charles Leclerc,Ferrari,2.0,4.0,Finished,12.0
63,2024,1,RUS,George Russell,Mercedes,3.0,5.0,Finished,10.0
4,2024,1,NOR,Lando Norris,McLaren,7.0,6.0,Finished,8.0
44,2024,1,HAM,Lewis Hamilton,Mercedes,9.0,7.0,Finished,6.0
81,2024,1,PIA,Oscar Piastri,McLaren,8.0,8.0,Finished,4.0
14,2024,1,ALO,Fernando Alonso,Aston Martin,6.0,9.0,Finished,2.0
18,2024,1,STR,Lance Stroll,Aston Martin,12.0,10.0,Finished,1.0


In [5]:
race["Position"] = pd.to_numeric(
    race["Position"],
    errors="coerce"
)

race["GridPosition"] = pd.to_numeric(race["GridPosition"],errors="coerce")

race["Top10"] = (race["Position"] <= 10).astype(int)

In [6]:
race["PositionsGained"] = (race["GridPosition"] - race["Position"]) #Фьючер для подсчета позиций, которые гонщик выиграл или проиграл относительно старта

In [7]:
all_races = []

schedule = fastf1.get_event_schedule(2024)

for _, event in schedule.iterrows():

    round_number = int(event["RoundNumber"])

    if round_number == 0:
        continue

    try:
        race = load_race_results(
            2024,
            round_number
        )

        race["EventName"] = event["EventName"]

        all_races.append(race)

        print(
            f"Loaded: {round_number} - {event['EventName']}"
        )

    except Exception as e:
        print(
            f"Failed: {round_number} - {event['EventName']}"
        )
        print(e)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Loaded: 1 - Bahrain Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Failed to align laps for driver

Loaded: 2 - Saudi Arabian Grand Prix


...

Loaded: 22 - Las Vegas Grand Prix


core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cach

Loaded: 23 - Qatar Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

Loaded: 24 - Abu Dhabi Grand Prix


In [8]:
season_2024 = pd.concat(
    all_races,
    ignore_index=True
)

In [9]:
season_2024.shape

(479, 25)

In [10]:
season_2024[
    [
        "EventName",
        "Abbreviation",
        "TeamName",
        "GridPosition",
        "Position"
    ]
].head(20)

,EventName,Abbreviation,TeamName,GridPosition,Position
0,Bahrain Grand Prix,VER,Red Bull Racing,1.0,1.0
1,Bahrain Grand Prix,PER,Red Bull Racing,5.0,2.0
2,Bahrain Grand Prix,SAI,Ferrari,4.0,3.0
3,Bahrain Grand Prix,LEC,Ferrari,2.0,4.0
4,Bahrain Grand Prix,RUS,Mercedes,3.0,5.0
5,Bahrain Grand Prix,NOR,McLaren,7.0,6.0
6,Bahrain Grand Prix,HAM,Mercedes,9.0,7.0
7,Bahrain Grand Prix,PIA,McLaren,8.0,8.0
8,Bahrain Grand Prix,ALO,Aston Martin,6.0,9.0
9,Bahrain Grand Prix,STR,Aston Martin,12.0,10.0


In [11]:
def load_season(year):
    all_races = []

    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():

        round_number = int(event["RoundNumber"])

        if round_number == 0:
            continue

        try:
            race = load_race_results(
                year,
                round_number
            )

            race["EventName"] = event["EventName"]

            all_races.append(race)

            print(
                f"{year} | Round {round_number} | {event['EventName']}"
            )

        except Exception as e:
            print(
                f"FAILED | {year} | Round {round_number}"
            )
            print(e)

    return pd.concat(
        all_races,
        ignore_index=True
    )

In [12]:
seasons = []

for year in range(2022, 2025):

    season = load_season(year)

    seasons.append(season)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

2022 | Round 1 | Bahrain Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

2022 | Round 2 | Saudi Arabian Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Driver  3: Ignoring late data f

...

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


FAILED | 2022 | Round 20
any API: 500 calls/h


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


FAILED | 2022 | Round 21
any API: 500 calls/h
FAILED | 2022 | Round 22
any API: 500 calls/h


RateLimitExceededError: any API: 500 calls/h

In [13]:
len(seasons)

1

In [14]:
[(i, s["Year"].iloc[0], len(s)) for i, s in enumerate(seasons)]

[(0, np.int64(2022), 140)]

In [15]:
df = pd.concat(seasons, ignore_index=True)
df.shape

(140, 25)

In [16]:
import os

os.makedirs("data/raw", exist_ok=True)

seasons[0].to_csv(
    "data/raw/f1_2022_partial.csv",
    index=False
)

In [17]:
seasons[0][
    ["Year", "Round", "EventName"]
].drop_duplicates()

,Year,Round,EventName
0,2022,1,Bahrain Grand Prix
20,2022,2,Saudi Arabian Grand Prix
40,2022,3,Australian Grand Prix
60,2022,4,Emilia Romagna Grand Prix
80,2022,5,Miami Grand Prix
100,2022,6,Spanish Grand Prix
120,2022,7,Monaco Grand Prix


In [20]:
season_2022_rest = load_season(
    2022,
    start_round=7
)

TypeError: load_season() got an unexpected keyword argument 'start_round'

In [21]:
import inspect

print(inspect.signature(load_season))

(year)


In [22]:
def load_season(year, start_round=1):
    all_races = []

    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():

        round_number = int(event["RoundNumber"])

        if round_number == 0:
            continue

        if round_number < start_round:
            continue

        try:
            print(
                f"Loading {year} | "
                f"Round {round_number} | "
                f"{event['EventName']}"
            )

            race = load_race_results(
                year,
                round_number
            )

            race["EventName"] = event["EventName"]

            all_races.append(race)

            print("SUCCESS")

        except Exception as e:
            print(
                f"FAILED | {year} | "
                f"Round {round_number}"
            )
            print(e)
            break

    if not all_races:
        return None

    return pd.concat(
        all_races,
        ignore_index=True
    )

In [23]:
import inspect

print(inspect.signature(load_season))

(year, start_round=1)


In [24]:
import inspect

print(inspect.signature(load_season))

(year, start_round=1)


In [25]:
seasons[0][
    ["Year", "Round", "EventName"]
].drop_duplicates()

,Year,Round,EventName
0,2022,1,Bahrain Grand Prix
20,2022,2,Saudi Arabian Grand Prix
40,2022,3,Australian Grand Prix
60,2022,4,Emilia Romagna Grand Prix
80,2022,5,Miami Grand Prix
100,2022,6,Spanish Grand Prix
120,2022,7,Monaco Grand Prix


In [26]:
season_2022_rest = load_season(
    2022,
    start_round=7
)

Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
    ~~~~~~~~~^^
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastf1\req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h


Loading 2022 | Round 7 | Monaco Grand Prix


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2022.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastf1\req.py", line 110, in send
    lim.limit()
    ~~~~~~~~~^^
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastf1\req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.3]


SUCCESS
Loading 2022 | Round 8 | Azerbaijan Grand Prix


req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


FAILED | 2022 | Round 8
any API: 500 calls/h


In [27]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Dataset builder initialized.")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

Dataset builder initialized.
Raw data directory: ../data/raw
Processed data directory: ../data/processed


In [28]:
BASE_COLUMNS = [
    "year",
    "round",
    "event_name",
    "circuit",
    "driver",
    "team",
    
    "grid_position",
    "finish_position",
    "race_points",
    
    "status",
    "laps",
    "positions_gained",
]

print("BASE COLUMNS")
print("=" * 50)

for column in BASE_COLUMNS:
    print(column)

BASE COLUMNS
year
round
event_name
circuit
driver
team
grid_position
finish_position
race_points
status
laps
positions_gained


In [29]:
def clean_race_results(df, year, round_number, event_name):
    """
    Clean raw FastF1 race results and convert them
    into a standardized dataframe.
    """
    
    data = df.copy()

    # Metadata
    data["year"] = year
    data["round"] = round_number
    data["event_name"] = event_name

    # Driver
    if "Abbreviation" in data.columns:
        data["driver"] = data["Abbreviation"]
    elif "FullName" in data.columns:
        data["driver"] = data["FullName"]
    else:
        data["driver"] = np.nan

    # Team
    if "TeamName" in data.columns:
        data["team"] = data["TeamName"]
    else:
        data["team"] = np.nan

    # Grid
    if "GridPosition" in data.columns:
        data["grid_position"] = pd.to_numeric(
            data["GridPosition"],
            errors="coerce"
        )
    else:
        data["grid_position"] = np.nan

    # Finish position
    if "Position" in data.columns:
        data["finish_position"] = pd.to_numeric(
            data["Position"],
            errors="coerce"
        )
    else:
        data["finish_position"] = np.nan

    # Points
    if "Points" in data.columns:
        data["race_points"] = pd.to_numeric(
            data["Points"],
            errors="coerce"
        )
    else:
        data["race_points"] = 0

    # Status
    if "Status" in data.columns:
        data["status"] = data["Status"]
    else:
        data["status"] = np.nan

    # Laps
    if "Laps" in data.columns:
        data["laps"] = pd.to_numeric(
            data["Laps"],
            errors="coerce"
        )
    else:
        data["laps"] = np.nan

    # Positions gained
    data["positions_gained"] = (
        data["grid_position"] - data["finish_position"]
    )

    # Circuit
    if "CircuitName" in data.columns:
        data["circuit"] = data["CircuitName"]
    else:
        data["circuit"] = event_name

    # Select only columns we need
    data = data[BASE_COLUMNS]

    return data

In [30]:
def inspect_dataset(df):
    print("DATASET OVERVIEW")
    print("=" * 60)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nColumns:")
    print(list(df.columns))

    print("\nMissing values:")
    missing = df.isna().sum()

    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst rows:")
    display(df.head())

In [32]:
print(type(seasons))
print(len(seasons))

for i, df in enumerate(seasons):
    print(f"\nDATASET {i}")
    print("=" * 50)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    
    if "Year" in df.columns:
        print("Years:", df["Year"].unique())
    
    if "Round" in df.columns:
        print("Rounds:", sorted(df["Round"].dropna().unique()))

<class 'list'>
1

DATASET 0
Rows: 140
Columns: 25
Years: [2022]
Rounds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [34]:
os.makedirs("data/raw", exist_ok=True)

seasons[0].to_csv(
    "data/raw/f1_2022_rounds_1_7.csv",
    index=False
)

print("Сохранено")
print(f"Rows saved: {len(seasons[0])}")

Сохранено
Rows saved: 140


In [37]:
saved_data = pd.read_csv("data/raw/f1_2022_rounds_1_7.csv")

print(saved_data.shape)
print(saved_data[["Year", "Round"]].drop_duplicates().sort_values("Round"))

(140, 25)
     Year  Round
0    2022      1
20   2022      2
40   2022      3
60   2022      4
80   2022      5
100  2022      6
120  2022      7


In [38]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Dataset builder initialized.")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")

Dataset builder initialized.
Raw data directory: ../data/raw
Processed data directory: ../data/processed


In [42]:
file_path = "data/raw/f1_2022_rounds_1_7.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

display(df.head())

Dataset loaded successfully!
Shape: (140, 25)


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Q1,Q2,Q3,Time,Status,Points,Laps,Year,Round,EventName
0,16,C LECLERC,LEC,leclerc,Ferrari,ed1c24,ferrari,Charles,Leclerc,Charles Leclerc,...,NaN,NaN,NaN,0 days 01:37:33.584000,Finished,26.0,57.0,2022,1,Bahrain Grand Prix
1,55,C SAINZ,SAI,sainz,Ferrari,ed1c24,ferrari,Carlos,Sainz,Carlos Sainz,...,NaN,NaN,NaN,0 days 00:00:05.598000,Finished,18.0,57.0,2022,1,Bahrain Grand Prix
2,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,...,NaN,NaN,NaN,0 days 00:00:09.675000,Finished,15.0,57.0,2022,1,Bahrain Grand Prix
3,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,...,NaN,NaN,NaN,0 days 00:00:11.211000,Finished,12.0,57.0,2022,1,Bahrain Grand Prix
4,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,b6babd,haas,Kevin,Magnussen,Kevin Magnussen,...,NaN,NaN,NaN,0 days 00:00:14.754000,Finished,10.0,57.0,2022,1,Bahrain Grand Prix


In [43]:
print("COLUMNS")
print("=" * 60)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

COLUMNS
 1. DriverNumber
 2. BroadcastName
 3. Abbreviation
 4. DriverId
 5. TeamName
 6. TeamColor
 7. TeamId
 8. FirstName
 9. LastName
10. FullName
11. HeadshotUrl
12. CountryCode
13. Position
14. ClassifiedPosition
15. GridPosition
16. Q1
17. Q2
18. Q3
19. Time
20. Status
21. Points
22. Laps
23. Year
24. Round
25. EventName


In [44]:
print("\nDATA TYPES")
print("=" * 60)

print(df.dtypes)


DATA TYPES
DriverNumber            int64
BroadcastName          object
Abbreviation           object
DriverId               object
TeamName               object
TeamColor              object
TeamId                 object
FirstName              object
LastName               object
FullName               object
HeadshotUrl            object
CountryCode            object
Position              float64
ClassifiedPosition     object
GridPosition          float64
Q1                    float64
Q2                    float64
Q3                    float64
Time                   object
Status                 object
Points                float64
Laps                  float64
Year                    int64
Round                   int64
EventName              object
dtype: object


In [45]:
print("\nFIRST 5 ROWS")
print("=" * 60)

display(df.head())


FIRST 5 ROWS


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Q1,Q2,Q3,Time,Status,Points,Laps,Year,Round,EventName
0,16,C LECLERC,LEC,leclerc,Ferrari,ed1c24,ferrari,Charles,Leclerc,Charles Leclerc,...,NaN,NaN,NaN,0 days 01:37:33.584000,Finished,26.0,57.0,2022,1,Bahrain Grand Prix
1,55,C SAINZ,SAI,sainz,Ferrari,ed1c24,ferrari,Carlos,Sainz,Carlos Sainz,...,NaN,NaN,NaN,0 days 00:00:05.598000,Finished,18.0,57.0,2022,1,Bahrain Grand Prix
2,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,...,NaN,NaN,NaN,0 days 00:00:09.675000,Finished,15.0,57.0,2022,1,Bahrain Grand Prix
3,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,...,NaN,NaN,NaN,0 days 00:00:11.211000,Finished,12.0,57.0,2022,1,Bahrain Grand Prix
4,20,K MAGNUSSEN,MAG,kevin_magnussen,Haas F1 Team,b6babd,haas,Kevin,Magnussen,Kevin Magnussen,...,NaN,NaN,NaN,0 days 00:00:14.754000,Finished,10.0,57.0,2022,1,Bahrain Grand Prix


In [46]:
clean_df = df[
    [
        "DriverId",
        "Abbreviation",
        "FullName",
        "TeamName",
        "Position",
        "GridPosition",
        "Q1",
        "Q2",
        "Q3",
        "Status",
        "Points",
        "Laps",
        "Year",
        "Round",
        "EventName",
    ]
].copy()

clean_df = clean_df.rename(columns={
    "DriverId": "driver_id",
    "Abbreviation": "driver",
    "FullName": "driver_name",
    "TeamName": "team",
    "Position": "finish_position",
    "GridPosition": "grid_position",
    "Q1": "q1",
    "Q2": "q2",
    "Q3": "q3",
    "Status": "status",
    "Points": "race_points",
    "Laps": "laps",
    "Year": "year",
    "Round": "round",
    "EventName": "event_name",
})

print("Чистый датасет")
print(f"Shape: {clean_df.shape}")

display(clean_df.head())

Чистый датасет
Shape: (140, 15)


,driver_id,driver,driver_name,team,finish_position,grid_position,q1,q2,q3,status,race_points,laps,year,round,event_name
0,leclerc,LEC,Charles Leclerc,Ferrari,1.0,1.0,NaN,NaN,NaN,Finished,26.0,57.0,2022,1,Bahrain Grand Prix
1,sainz,SAI,Carlos Sainz,Ferrari,2.0,3.0,NaN,NaN,NaN,Finished,18.0,57.0,2022,1,Bahrain Grand Prix
2,hamilton,HAM,Lewis Hamilton,Mercedes,3.0,5.0,NaN,NaN,NaN,Finished,15.0,57.0,2022,1,Bahrain Grand Prix
3,russell,RUS,George Russell,Mercedes,4.0,9.0,NaN,NaN,NaN,Finished,12.0,57.0,2022,1,Bahrain Grand Prix
4,kevin_magnussen,MAG,Kevin Magnussen,Haas F1 Team,5.0,7.0,NaN,NaN,NaN,Finished,10.0,57.0,2022,1,Bahrain Grand Prix


In [47]:
print("MISSING VALUES")
print("=" * 50)

missing = clean_df.isna().sum()

print(missing[missing > 0])

MISSING VALUES
finish_position      1
grid_position        1
q1                 140
q2                 140
q3                 140
dtype: int64


In [49]:
print("DATA TYPES")
print("-" * 50)

print(clean_df.dtypes)

DATA TYPES
--------------------------------------------------
driver_id           object
driver              object
driver_name         object
team                object
finish_position    float64
grid_position      float64
q1                 float64
q2                 float64
q3                 float64
status              object
race_points        float64
laps               float64
year                 int64
round                int64
event_name          object
dtype: object


In [51]:
def download_race(year, round_number):
    filename = f"{RAW_DIR}/f1_{year}_round_{round_number}.csv"

    # Если уже скачана — пропускаем
    if os.path.exists(filename):
        print(f"SKIP | {year} Round {round_number} | already exists")
        return pd.read_csv(filename)

    print(f"LOADING | {year} Round {round_number}")

    session = fastf1.get_session(year, round_number, "R")
    session.load()

    results = session.results.copy()

    results["Year"] = year
    results["Round"] = round_number

    results.to_csv(filename, index=False)

    print(f"SAVED | {filename} | {len(results)} rows")

    return results

In [52]:
test_race = download_race(2022, 8)

print(test_race.shape)
display(test_race.head())

LOADING | 2022 Round 8


core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached dat

SAVED | ../data/raw/f1_2022_round_8.csv | 20 rows
(20, 24)


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Year,Round
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,1e5bc6,red_bull,Max,Verstappen,Max Verstappen,...,3.0,NaT,NaT,NaT,0 days 01:34:05.941000,Finished,25.0,51.0,2022,8
11,11,S PEREZ,PER,perez,Red Bull Racing,1e5bc6,red_bull,Sergio,Perez,Sergio Perez,...,2.0,NaT,NaT,NaT,0 days 00:00:20.823000,Finished,19.0,51.0,2022,8
63,63,G RUSSELL,RUS,russell,Mercedes,6cd3bf,mercedes,George,Russell,George Russell,...,5.0,NaT,NaT,NaT,0 days 00:00:45.995000,Finished,15.0,51.0,2022,8
44,44,L HAMILTON,HAM,hamilton,Mercedes,6cd3bf,mercedes,Lewis,Hamilton,Lewis Hamilton,...,7.0,NaT,NaT,NaT,0 days 00:01:11.679000,Finished,12.0,51.0,2022,8
10,10,P GASLY,GAS,gasly,AlphaTauri,4e7c9b,alphatauri,Pierre,Gasly,Pierre Gasly,...,6.0,NaT,NaT,NaT,0 days 00:01:17.299000,Finished,10.0,51.0,2022,8


In [54]:
import time
for round_number in range(9, 23):
    try:
        download_race(2022, round_number)
        time.sleep(3)

    except Exception as e:
        print(f"FAILED | 2022 Round {round_number}")
        print(e)
        break

SKIP | 2022 Round 9 | already exists


core           INFO 	Loading data for British Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


LOADING | 2022 Round 10


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Driver  1: Ignoring late data f

SAVED | ../data/raw/f1_2022_round_10.csv | 20 rows


...

SAVED | ../data/raw/f1_2022_round_21.csv | 20 rows


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


LOADING | 2022 Round 22


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

SAVED | ../data/raw/f1_2022_round_22.csv | 20 rows


In [55]:
RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)


def download_race(year, round_number):
    filename = f"{RAW_DIR}/f1_{year}_round_{round_number}.csv"

    if os.path.exists(filename):
        print(f"SKIP | {year} Round {round_number} | already exists")
        return pd.read_csv(filename)

    print(f"LOADING | {year} Round {round_number}")

    session = fastf1.get_session(year, round_number, "R")
    session.load()

    results = session.results.copy()

    results["Year"] = year
    results["Round"] = round_number

    results.to_csv(filename, index=False)

    print(f"SAVED | {filename} | {len(results)} rows")

    return results

In [56]:
test_race = download_race(2023, 1)

print(test_race.shape)
display(test_race.head())

LOADING | 2023 Round 1


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

SAVED | data/raw/f1_2023_round_1.csv | 20 rows
(20, 24)


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Year,Round
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,NaT,NaT,NaT,0 days 01:33:56.736000,Finished,25.0,57.0,2023,1
11,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,...,2.0,NaT,NaT,NaT,0 days 00:00:11.987000,Finished,18.0,57.0,2023,1
14,14,F ALONSO,ALO,alonso,Aston Martin,358C75,aston_martin,Fernando,Alonso,Fernando Alonso,...,5.0,NaT,NaT,NaT,0 days 00:00:38.637000,Finished,15.0,57.0,2023,1
55,55,C SAINZ,SAI,sainz,Ferrari,F91536,ferrari,Carlos,Sainz,Carlos Sainz,...,4.0,NaT,NaT,NaT,0 days 00:00:48.052000,Finished,12.0,57.0,2023,1
44,44,L HAMILTON,HAM,hamilton,Mercedes,6CD3BF,mercedes,Lewis,Hamilton,Lewis Hamilton,...,7.0,NaT,NaT,NaT,0 days 00:00:50.977000,Finished,10.0,57.0,2023,1


In [57]:
for round_number in range(2, 23):
    try:
        download_race(2023, round_number)
        time.sleep(4)

    except Exception as e:
        print(f"FAILED | 2023 Round {round_number}")
        print(e)
        break

core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


LOADING | 2023 Round 2


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

SAVED | data/raw/f1_2023_round_2.csv | 20 rows


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


...

SAVED | data/raw/f1_2023_round_21.csv | 20 rows


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


LOADING | 2023 Round 22


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

SAVED | data/raw/f1_2023_round_22.csv | 20 rows


In [58]:
files = sorted(
    f for f in os.listdir(RAW_DIR)
    if f.startswith("f1_2023_round_") and f.endswith(".csv")
)

print(f"Downloaded races: {len(files)}")

for f in files:
    print(f)

Downloaded races: 22
f1_2023_round_1.csv
f1_2023_round_10.csv
f1_2023_round_11.csv
f1_2023_round_12.csv
f1_2023_round_13.csv
f1_2023_round_14.csv
f1_2023_round_15.csv
f1_2023_round_16.csv
f1_2023_round_17.csv
f1_2023_round_18.csv
f1_2023_round_19.csv
f1_2023_round_2.csv
f1_2023_round_20.csv
f1_2023_round_21.csv
f1_2023_round_22.csv
f1_2023_round_3.csv
f1_2023_round_4.csv
f1_2023_round_5.csv
f1_2023_round_6.csv
f1_2023_round_7.csv
f1_2023_round_8.csv
f1_2023_round_9.csv
